In [1]:
# -*- coding: utf-8 -*-
"""
============================================================================
 preprocess_interaction_2class_split.py — SAPA: MERL 6-kelas -> 2-kelas
 + split proper (80% dev-pool / 20% test tersisih, by SUBJEK)
============================================================================
 TIDAK ekstrak ulang pose dari video (mahal, tidak perlu). Cuma:
   1. Relabel: kelas asli {4=inspect_product, 5=inspect_shelf} -> 1 (inspecting)
              kelas asli {0,1,2,3} -> 0 (other)
   2. Split by SUBJEK (bukan acak!) jadi:
        - dev_pool (80%) -> akan dipakai 5-fold CV di skrip training
        - test_holdout (20%) -> TIDAK PERNAH disentuh sampai evaluasi akhir
   3. Simpan sebagai 2 file .npz terpisah, supaya training TIDAK BISA
      "mengintip" test set secara tidak sengaja.

 INPUT : merl.npz (sudah ada, HASIL EKSTRAKSI LAMA — tidak diulang)
 OUTPUT (folder BARU, tidak menimpa): preprocessed_interaction_2class/
   - interaction_2class_devpool.npz   (80%, utk 5-fold CV)
   - interaction_2class_test.npz      (20%, held-out, JANGAN dipakai sampai akhir)
   - split_info.json                  (daftar subjek per split, utk audit)

 Cepat (numpy saja, CPU cukup, tidak perlu GPU/A100 sama sekali untuk file ini).
 Copy tiap "# ===== CELL N =====" ke satu cell Colab.
============================================================================
"""

# ===== CELL 1: Mount + konfigurasi =====
from google.colab import drive
drive.mount('/content/drive')

import os, json, numpy as np

BASE       = "/content/drive/MyDrive/Belajar AI/Compfest/Dataset"
NPZ_IN     = f"{BASE}/preprocessed_2/merl.npz"     # data LAMA, tidak diulang ekstraksinya
OUT_DIR    = f"{BASE}/preprocessed_interaction_2class"
os.makedirs(OUT_DIR, exist_ok=True)

# kelas asli MERL: 0=background 1=reach 2=retract 3=hand_in_shelf
#                  4=inspect_product 5=inspect_shelf
INSPECT_ORIGINAL_IDX = [4, 5]
TEST_FRACTION = 0.20
SEED = 42

np.random.seed(SEED)




Mounted at /content/drive


In [2]:
# ===== CELL 2: Load + relabel jadi 2 kelas =====
d = np.load(NPZ_IN, allow_pickle=True)
kp   = d["keypoints"]                # [N,45,17,3]
y6   = d["labels"].astype(np.int64)  # label asli 0..5
meta = d["meta"].astype(str)         # format "{subjek}_{sesi}"

N = kp.shape[0]
print(f"Total window (6-kelas asli): {N}")
print("Distribusi 6-kelas:", {c: int((y6==c).sum()) for c in range(6)})

# RELABEL: 2 kelas
y2 = np.isin(y6, INSPECT_ORIGINAL_IDX).astype(np.int64)   # 1=inspecting, 0=other
CLASS_NAMES_2 = ["other", "inspecting"]
print("\nDistribusi 2-kelas (SETELAH relabel):")
print(f"  other      : {int((y2==0).sum())}")
print(f"  inspecting : {int((y2==1).sum())}")

def subject_of(m):
    return m.split("_")[0]   # "10_1" -> "10"

subjects = np.array([subject_of(m) for m in meta])
unique_subjects = sorted(set(subjects))
print(f"\nJumlah subjek unik: {len(unique_subjects)}")




Total window (6-kelas asli): 14165
Distribusi 6-kelas: {0: 3113, 1: 1666, 2: 1812, 3: 1209, 4: 2709, 5: 3656}

Distribusi 2-kelas (SETELAH relabel):
  other      : 7800
  inspecting : 6365

Jumlah subjek unik: 41


In [3]:
# ===== CELL 3: Split 80/20 BY SUBJEK (bukan acak per-window!) =====
# Kenapa by subjek: kalau split acak per-window, window dari subjek yang sama
# bisa masuk dev_pool DAN test -> bocor (model "mengenal" gaya gerak subjek
# itu dari dev_pool, lalu diuji di window lain subjek yang sama = tidak jujur).
rng = np.random.RandomState(SEED)
shuffled_subjects = unique_subjects.copy()
rng.shuffle(shuffled_subjects)

n_test_subj = max(1, int(round(len(shuffled_subjects) * TEST_FRACTION)))
test_subjects = set(shuffled_subjects[:n_test_subj])
dev_subjects  = set(shuffled_subjects[n_test_subj:])

print(f"Subjek test  ({len(test_subjects)}): {sorted(test_subjects)}")
print(f"Subjek dev   ({len(dev_subjects)}): {sorted(dev_subjects)}")

is_test = np.array([s in test_subjects for s in subjects])
is_dev  = ~is_test

print(f"\nWindow dev_pool : {is_dev.sum()}")
print(f"Window test     : {is_test.sum()}")

# cek keseimbangan kelas tidak terlalu timpang di test (informatif, bukan wajib sama persis)
print("\nDistribusi 2-kelas di dev_pool :", {CLASS_NAMES_2[c]: int((y2[is_dev]==c).sum()) for c in [0,1]})
print("Distribusi 2-kelas di test     :", {CLASS_NAMES_2[c]: int((y2[is_test]==c).sum()) for c in [0,1]})

assert set(subjects[is_dev]).isdisjoint(set(subjects[is_test])), "BOCOR: ada subjek di dua split!"
print("\n✓ Verifikasi: tidak ada subjek yang bocor antara dev_pool dan test.")




Subjek test  (8): [np.str_('13'), np.str_('17'), np.str_('21'), np.str_('27'), np.str_('31'), np.str_('32'), np.str_('8'), np.str_('9')]
Subjek dev   (33): [np.str_('1'), np.str_('10'), np.str_('11'), np.str_('12'), np.str_('14'), np.str_('15'), np.str_('16'), np.str_('18'), np.str_('19'), np.str_('2'), np.str_('20'), np.str_('22'), np.str_('23'), np.str_('24'), np.str_('25'), np.str_('26'), np.str_('28'), np.str_('29'), np.str_('3'), np.str_('30'), np.str_('33'), np.str_('34'), np.str_('35'), np.str_('36'), np.str_('37'), np.str_('38'), np.str_('39'), np.str_('4'), np.str_('40'), np.str_('41'), np.str_('5'), np.str_('6'), np.str_('7')]

Window dev_pool : 10948
Window test     : 3217

Distribusi 2-kelas di dev_pool : {'other': 6017, 'inspecting': 4931}
Distribusi 2-kelas di test     : {'other': 1783, 'inspecting': 1434}

✓ Verifikasi: tidak ada subjek yang bocor antara dev_pool dan test.


In [4]:
# ===== CELL 4: Simpan 2 file terpisah =====
np.savez_compressed(
    f"{OUT_DIR}/interaction_2class_devpool.npz",
    keypoints=kp[is_dev], labels=y2[is_dev], meta=meta[is_dev],
    class_names=np.array(CLASS_NAMES_2), fps=15, source="merl_2class_devpool"
)
np.savez_compressed(
    f"{OUT_DIR}/interaction_2class_test.npz",
    keypoints=kp[is_test], labels=y2[is_test], meta=meta[is_test],
    class_names=np.array(CLASS_NAMES_2), fps=15, source="merl_2class_test_HOLDOUT"
)

split_info = {
    "seed": SEED,
    "test_fraction_target": TEST_FRACTION,
    "n_subjects_total": len(unique_subjects),
    "dev_subjects": sorted(dev_subjects),
    "test_subjects": sorted(test_subjects),
    "n_window_dev": int(is_dev.sum()),
    "n_window_test": int(is_test.sum()),
    "class_names": CLASS_NAMES_2,
    "note": "Kelas 'inspecting' = gabungan inspect_product(4) + inspect_shelf(5) MERL asli. "
            "'other' = reach(1)+retract(2)+hand_in_shelf(3)+background(0). "
            "Split BY SUBJEK, bukan acak per-window, utk cegah kebocoran."
}
with open(f"{OUT_DIR}/split_info.json", "w") as f:
    json.dump(split_info, f, indent=2)

print(f"\n=== SELESAI ===")
print(f"Tersimpan:")
print(f"  {OUT_DIR}/interaction_2class_devpool.npz  ({is_dev.sum()} window, utk 5-fold CV)")
print(f"  {OUT_DIR}/interaction_2class_test.npz     ({is_test.sum()} window, HELD-OUT, jangan disentuh)")
print(f"  {OUT_DIR}/split_info.json                 (audit trail split)")
print(f"\n>> LANGKAH SELANJUTNYA: skrip training pakai devpool.npz utk 5-fold CV.")
print(f">> test.npz HANYA dibuka SEKALI di akhir, setelah model final terpilih.")


=== SELESAI ===
Tersimpan:
  /content/drive/MyDrive/Belajar AI/Compfest/Dataset/preprocessed_interaction_2class/interaction_2class_devpool.npz  (10948 window, utk 5-fold CV)
  /content/drive/MyDrive/Belajar AI/Compfest/Dataset/preprocessed_interaction_2class/interaction_2class_test.npz     (3217 window, HELD-OUT, jangan disentuh)
  /content/drive/MyDrive/Belajar AI/Compfest/Dataset/preprocessed_interaction_2class/split_info.json                 (audit trail split)

>> LANGKAH SELANJUTNYA: skrip training pakai devpool.npz utk 5-fold CV.
>> test.npz HANYA dibuka SEKALI di akhir, setelah model final terpilih.
